# Integrated Replay-Free CLM Kill Test 001 — Development Smoke

This notebook runs only the registered development seed. It prepares the pinned KT001 snapshot with explicit historical-prefix overlap accounting, runs all five causal arms, aggregates the development decision, and Git-publishes lightweight evidence. Formal execution remains blocked until a separately reviewed `IMPLEMENTATION_LOCK.json` is sealed.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BRANCH = 'codex/integrated-replay-free-clm-kill-test-001'
REPO = Path('/kaggle/working/mini-cells')
CHECKPOINT_DIR = Path('/kaggle/working/native-clm-v0-m1')
CHECKPOINT = CHECKPOINT_DIR / 'final-model.pt'
DATA = Path('/kaggle/working/kt001-data')
OUT = REPO / 'artifacts/experiments/integrated-replay-free-clm-kill-test-001'

def run(cmd, check=True, env=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check, env=env)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/ArcheLabs/mini-cells.git', REPO])
else:
    os.chdir(REPO)
    run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'])
    run(['git', 'checkout', BRANCH])
    run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'])
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'])
HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('HEAD:', HEAD)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import auth_check
import torch

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'Missing HF_TOKEN'
assert os.environ['GITHUB_TOKEN'], 'Missing GITHUB_TOKEN'
assert torch.cuda.is_available(), 'CUDA required for KT001 development run'
GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])
auth_check('archelabsxyz/native-clm-v0', repo_type='model', token=os.environ['HF_TOKEN'])
print('Secrets + HF read preflight: PASS')

In [ ]:
run([
    sys.executable, 'scripts/research/fetch_native_clm_v0_m1_checkpoint.py',
    '--repo-id', 'archelabsxyz/native-clm-v0',
    '--filename', 'final-model.pt',
    '--expected-sha256', '91cc66f744c97e50105acbb7cdc328a95cb87a32c49baf5b0d6e462d4d4c4c7f',
    '--output', CHECKPOINT,
])
print((CHECKPOINT_DIR / 'provenance.json').read_text())

In [ ]:
# Prefer post-history records from exact pinned revisions. If an exact split cannot
# provide a same-size disjoint replacement, only the unavoidable shortfall is reused
# and the exact overlap is recorded in manifest.json.
run([
    sys.executable, 'scripts/research/prepare_integrated_replay_free_clm_kt001_data.py',
    '--output-dir', DATA,
])
manifest = json.loads((DATA / 'manifest.json').read_text())
print(json.dumps({
    'format': manifest['format'],
    'stream': manifest['stream'],
    'selector_salt': manifest['selector_salt'],
    'fully_disjoint': manifest['fully_disjoint_from_historical_prefixes'],
    'historical_prefix_reused_documents': manifest['historical_prefix_reused_documents_accounted'],
    'dataset_revisions': manifest['dataset_revisions'],
}, indent=2))

In [ ]:
registry = json.loads(Path(
    'research/experiments/04-continual-learning-core/integrated-replay-free-clm-kill-test-001/SEEDS.json'
).read_text())
DEV_SEED = int(registry['development'][0])
LOCK = Path('research/experiments/04-continual-learning-core/integrated-replay-free-clm-kill-test-001/IMPLEMENTATION_LOCK.json')
assert DEV_SEED not in set(map(int, registry['formal']))
print('Development seed:', DEV_SEED)
print('Formal lock exists:', LOCK.exists())

In [ ]:
ARMS = ['unsafe', 'write_transaction_only', 'read_history_only', 'full_no_replay', 'matched_replay_oracle']

def arm_command(arm):
    return [
        sys.executable, 'scripts/research/run_integrated_replay_free_clm_kt001.py',
        '--checkpoint', str(CHECKPOINT),
        '--data-dir', str(DATA),
        '--output-dir', str(OUT),
        '--seed', str(DEV_SEED),
        '--arm', arm,
        '--device', 'cuda',
    ]

def publish_partial_failure():
    result = run([
        sys.executable, 'scripts/research/publish_integrated_replay_free_clm_kt001.py',
        '--seed', str(DEV_SEED),
        '--branch', BRANCH,
        '--output-dir', OUT,
        '--checkpoint-provenance', CHECKPOINT_DIR / 'provenance.json',
        '--data-manifest', DATA / 'manifest.json',
    ], check=False)
    print('Partial failure publication return code:', result.returncode)

def launch_pair(pair):
    processes = []
    for index, arm in enumerate(pair):
        gpu = index % max(1, GPU_COUNT)
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        cmd = arm_command(arm)
        print(f'+ [GPU {gpu}]', ' '.join(cmd), flush=True)
        processes.append((arm, subprocess.Popen(cmd, env=env)))
    failures = []
    for arm, process in processes:
        code = process.wait()
        if code != 0:
            failures.append((arm, code))
    if failures:
        publish_partial_failure()
        raise RuntimeError(f'KT001 development arm failures: {failures}')

if GPU_COUNT >= 2:
    launch_pair(ARMS[0:2])
    launch_pair(ARMS[2:4])
    launch_pair(ARMS[4:5])
else:
    for arm in ARMS:
        launch_pair([arm])
print('All development arms completed.')

In [ ]:
run([
    sys.executable, 'scripts/research/aggregate_integrated_replay_free_clm_kt001.py',
    '--output-dir', OUT,
    '--seed', str(DEV_SEED),
])
development = json.loads((OUT / f'seed-{DEV_SEED}' / 'seed-decision.json').read_text())
print(json.dumps({
    'seed': development['seed'],
    'classification': development['classification'],
    'mechanics_gates': development['mechanics_gates'],
    'oracle_gates': development['oracle_gates'],
    'scientific_gates': development['scientific_gates'],
}, indent=2))

In [ ]:
# Development: large checkpoints remain local by default; lightweight evidence is
# committed and pushed after the seed decision is written.
run([
    sys.executable, 'scripts/research/publish_integrated_replay_free_clm_kt001.py',
    '--seed', str(DEV_SEED),
    '--branch', BRANCH,
    '--output-dir', OUT,
    '--checkpoint-provenance', CHECKPOINT_DIR / 'provenance.json',
    '--data-manifest', DATA / 'manifest.json',
])
print('Published KT001 development classification:', development['classification'])

## Formal boundary

Do **not** run formal seeds from this development notebook. After development mechanics are accepted, the repository must seal `IMPLEMENTATION_LOCK.json`; a separate formal entry point may then read the formal seed registry dynamically.